# Agent liniowy — przepływ krok po kroku

Ten notebook najpierw pokazuje **jak działa** agent liniowy — uruchamia po kolei
umiejętności z `src/skills/*` i pokazuje wynik **każdego etapu** (to samo, co robi
`agent_linear/pipeline.py`, ale interaktywnie). Potem **ewaluuje** wygenerowaną
apelację: pokrycie (raz z drukiem + 5× na powtarzalność i koszt).

> To **nie** jest baseline (jeden wielki prompt) — to agent dzielący pracę na kroki.
> Pierwsza komórka wczytuje konfigurację LLM z pliku `.env`.

In [1]:
import os

# Konfiguracja LLM (klucz, model) wczytana z pliku .env — nic na sztywno w komórce.
from dotenv import load_dotenv

load_dotenv()
os.environ["LLM_MODEL"] = "gpt-5.4-mini"  # w notebookach tani model — to tylko demo
print("model:", os.environ.get("LLM_MODEL"), "| klucz z .env:", bool(os.environ.get("LLM_API_KEY")))

from src.loader import load_all
from src.sources import prepare_input_texts
from src.skills.file_description.main import generate_file_description
from src.skills.tasks.main import generate_tasks
from src.skills.make_task.main import make_task
from src.skills.strategy.main import generate_strategy
from src.skills.document.main import generate_document

pipeline_usages = {}  # zużycie tokenów per etap (do policzenia kosztu)

GOAL = "Wygeneruj apelację z perspektywy obrony"
GOAL

model: gpt-5.4-mini | klucz z .env: True


'Wygeneruj apelację z perspektywy obrony'

## 0. Wczytanie akt

In [2]:
documents = load_all()
print(f"Wczytano {len(documents)} dokumentów")
for d in documents:
    print(" -", d.filename)

Wczytano 16 dokumentów
 - 02_Notatka_urzędowa_z_interwencji_dotyczącej_nietrzeźwego_kierowcy.pdf
 - 03_Protokół_z_badania_stanu_trzeźwości_analizatorem_wydechu.pdf
 - 04_Postanowienie_o_wszczęciu_dochodzenia.pdf
 - 05_Protokół_przesłuchania_świadka.pdf
 - 06_Protokół_przesłuchania_świadka.pdf
 - 07_Protokół_przesłuchania_świadka.pdf
 - 08_Postanowienie_o_przedstawieniu_zarzutów.pdf
 - 09_Protokół_przesłuchania_podejrzanego.pdf
 - 10_Protokół_przesłuchania_podejrzanego.pdf
 - 11_Akt_oskarżenia.pdf
 - 12_Protokół_rozprawy_głównej.pdf
 - 13_Odpowiedź_na_wezwanie_Sądu_Rejonowego_dotyczące_zarządzania_cmentarzem.pdf
 - 14_Opinia_biegłego_sądowego_z_zakresu_toksykologii_sądowej.pdf
 - 15_Protokół_rozprawy_głównej.pdf
 - 16_Wyrok.pdf
 - 17_Wniosek_o_sporządzenie_uzasadnienia.pdf


In [3]:
documents[0]

Document(filename='02_Notatka_urzędowa_z_interwencji_dotyczącej_nietrzeźwego_kierowcy.pdf', text='Komisariat Policji w Balicach Balice, 31 grudnia 2024 r.\nsierż. Wiesław Wilk\nNOTATKA URZĘDOWA\nW dniu datowania pełniłem służbę w obchodzie z post. Sylwią Sikorą na terenie\nKP Balice.\nOk. godz. 10.40 z polecenia dyżurnego KP Balice udaliśmy się do miejscowości\nSzczyglice w rejon cmentarza, gdzie według zgłoszenia kierujący pojazdem marki Jeep\nWrangler może być nietrzeźwy. Na miejscu zastano zgłaszającego: Karola Kota (Kot), który\noświadczył, że widział, jak kierujący pojazdem na terenie należącym do cmentarza usiłował\nzawrócić pojazd w alejce. Zgłaszający stwierdził, że kierowca prawdopodobnie jest\nnietrzeźwy. Następnie Karol Kot wskazał nam pojazd i udał się na pogrzeb. W pojeździe\nwskazanym przez Karola Kota na miejscu kierowcy zastano mężczyznę, od którego\nwyczuwalna była silna woń alkoholu. Wyciągnięto kluczyki ze stacyjki. Mężczyzną okazał\nsię być:\nDaniel Dzik, s. Jana i 

## 1. `generate_file_description` — opis każdego dokumentu

Surowe akta → zwięzłe, ukierunkowane na cel opisy ("mapa" sprawy).

In [4]:
from src.llm import track_usage

with track_usage() as _u:
    described = [generate_file_description(doc, GOAL) for doc in documents]
pipeline_usages["file_description"] = _u

In [5]:
described[0]

DescribedFile(title='NOTATKA URZĘDOWA', description='Dokument zawiera szczegółowy opis interwencji Policji wobec Daniela Dzika, którego zatrzymano jako kierującego pojazdem Jeep Wrangler nr rej. KRA 56789 po zgłoszeniu o możliwej nietrzeźwości. W notatce wskazano wyniki badań na zawartość alkoholu w wydychanym powietrzu: 1,63 mg/l, 1,57 mg/l, następnie 1,34 mg/l i 1,22 mg/l, a także informację o elektronicznym zatrzymaniu prawa jazdy nr 00123/00/1234567. Dokument jest istotny dla obrony, ponieważ zawiera dane o przebiegu czynności, czasie badań, oświadczeniu Daniela Dzika o spożyciu alkoholu oraz okolicznościach przekazania pojazdu i osoby pod opiekę żony.', name='02_Notatka_urzędowa_z_interwencji_dotyczącej_nietrzeźwego_kierowcy.pdf')

## 2. `generate_tasks` — plan analizy

Każdy krok wskazuje, **które** dokumenty są mu potrzebne (`input`).

In [6]:
with track_usage() as _u:
    tasks = generate_tasks(GOAL, described)
pipeline_usages["generate_tasks"] = _u

for i, t in enumerate(tasks.thoughts, 1):
    print(f"[{i}] {t.action}")
    print(f"    dokumenty: {t.input}")
    print(f"    rozumowanie: {t.reasoning}")
    print()

[1] Przeanalizować wyrok i ustalić pełny zakres rozstrzygnięcia wobec Daniela Dzika, w tym kary, środki karne, obowiązki i koszty, a także potwierdzić, że obrona zainicjowała etap odwoławczy przez wniosek o uzasadnienie.
    dokumenty: ['16_Wyrok.pdf', '17_Wniosek_o_sporządzenie_uzasadnienia.pdf']
    rozumowanie: Najpierw trzeba ustalić treść rozstrzygnięcia końcowego, zakres skazania, wymierzone kary i środki karne oraz to, jakie elementy wyroku mogą być zaskarżone w apelacji obrony. Bez tego nie da się dobrać właściwych zarzutów ani określić, czy apelacja ma dotyczyć całości czy tylko części wyroku.

[2] Zrekonstruować przebieg zdarzenia, ustalenia sądu i rozbieżności dowodowe dotyczące prowadzenia pojazdu, stanu nietrzeźwości oraz zniszczenia ławki, ze szczególnym uwzględnieniem opinii biegłego i zeznań świadków.
    dokumenty: ['12_Protokół_rozprawy_głównej.pdf', '14_Opinia_biegłego_sądowego_z_zakresu_toksykologii_sądowej.pdf', '05_Protokół_przesłuchania_świadka.pdf', '06_Protokół

## 3. `make_task` — wykonanie kroków na wybranych dokumentach

Sedno: `prepare_input_texts` daje krokowi **tylko** wskazane dokumenty (selektywny kontekst).

In [7]:
from src.llm import track_usage

task_outputs = []
with track_usage() as _u:
    for i, task in enumerate(tasks.thoughts, 1):
        sources_text = prepare_input_texts(documents, task.input)
        out = make_task(GOAL, task, sources_text)
        task_outputs.append(out)
pipeline_usages["make_task"] = _u

In [8]:
task_outputs[0]

TaskOutput(output='Wyrok wobec Daniela Dzika obejmuje dwa przypisane czyny i następujące rozstrzygnięcia: (1) za czyn z art. 178a § 1 k.k. – kara 200 stawek dziennych grzywny, przy stawce 20 zł, czyli łącznie 4 000 zł; (2) środek karny zakazu prowadzenia wszelkich pojazdów mechanicznych na 5 lat; (3) środek karny świadczenia pieniężnego 6 000 zł na rzecz Funduszu Pomocy Pokrzywdzonym oraz Pomocy Postpenitencjarnej; (4) przepadek na rzecz Skarbu Państwa samochodu Jeep Wrangler nr rej. KRA 56789, rok prod. 2010, nr VIN W1K1234567Z654321; (5) za czyn z art. 288 § 1 k.k. – kara 200 stawek dziennych grzywny po 20 zł, czyli 4 000 zł; (6) obowiązek naprawienia szkody przez zapłatę 7 000 zł na rzecz Ryszarda Rysia; (7) kara łączna grzywny 350 stawek dziennych po 20 zł, czyli 7 000 zł; (8) zaliczenie na poczet zakazu prowadzenia pojazdów okresu zatrzymania prawa jazdy od 31 grudnia 2024 r.; (9) koszty procesu: 1 987 zł wydatków oraz 700 zł opłaty. Wniosek o sporządzenie uzasadnienia został skut

## 4. `generate_strategy` — wybór strategii

In [9]:
from src.llm import track_usage

with track_usage() as _u:
    strategy = generate_strategy(GOAL, task_outputs)
pipeline_usages["generate_strategy"] = _u

In [10]:
strategy

Strategy(definition_of_success='Sukces w tym zadaniu oznacza sporządzenie apelacji, która w sposób przekonujący wskazuje na błędy sądu I instancji, zwłaszcza naruszenie art. 7 k.p.k., art. 5 § 2 k.p.k. oraz błąd w ustaleniach faktycznych co do prowadzenia pojazdu i czasu spożycia alkoholu, a także eksponuje nadmierną surowość zakazu prowadzenia pojazdów, przepadku samochodu i kary łącznej. Apelacja powinna być oparta na pełnym obrazie sprawy, ale z wyraźnym priorytetem dla zarzutów dotyczących czynu z art. 178a § 1 k.k., ponieważ tam istnieje realna przestrzeń do wzruszenia wyroku.', strategies='Możliwe są trzy główne strategie. Po pierwsze, strategia ofensywna: skoncentrować apelację na wykazaniu, że nie udowodniono ponad rozsądną wątpliwość, iż oskarżony prowadził pojazd w stanie nietrzeźwości, akcentując brak bezpośredniej obserwacji wjazdu, zamkniętą bramę, wąską alejkę, niejednoznaczność zeznań świadków oraz ograniczenia opinii biegłego. Po drugie, strategia mieszana: kwestionować

## 5. `generate_document` — gotowa apelacja

In [11]:
from src.llm import track_usage

with track_usage() as _u:
    document = generate_document(GOAL, strategy, task_outputs)
pipeline_usages["generate_document"] = _u

print(document.tekst)

APELACJA

w imieniu oskarżonego Daniela Dzika

od wyroku Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie II Wydziału Karnego, wydanego w sprawie o sygn. akt II K 25/25

Działając jako obrońca oskarżonego Daniela Dzika, na podstawie art. 425 § 1 i 2 k.p.k., art. 444 k.p.k. oraz art. 427 § 1 i 2 k.p.k., zaskarżam powyższy wyrok w całości na korzyść oskarżonego.

Zaskarżonemu wyrokowi zarzucam:

I. na podstawie art. 438 pkt 3 k.p.k. błąd w ustaleniach faktycznych przyjętych za podstawę orzeczenia, mający istotny wpływ na jego treść, polegający na bezpodstawnym przyjęciu, że oskarżony Daniel Dzik dopuścił się czynu z art. 178a § 1 k.k., tj. prowadził pojazd mechaniczny w ruchu lądowym w stanie nietrzeźwości, podczas gdy prawidłowa, wszechstronna ocena całokształtu materiału dowodowego prowadzi do wniosku przeciwnego, a co najmniej do powstania nieusuwalnych wątpliwości, których sąd I instancji nie rozstrzygnął na korzyść oskarżonego, w szczególności wobec:

1) braku bezpośredniego dowodu 

## 5b. Koszt wygenerowania apelacji (cały pipeline)

Sumujemy zużycie tokenów ze wszystkich etapów agenta — koszt per etap i łącznie
(porównaj z baseline, gdzie to jeden wielki prompt).

In [12]:
from src.cost import estimate_cost

model = os.environ["LLM_MODEL"]
total_cost, total_calls, total_tok = 0.0, 0, 0
print(f"Koszt pipeline'u (model: {model}) — per etap:\n")
for name, u in pipeline_usages.items():
    c = estimate_cost(u, model)
    total_cost += c
    total_calls += u.calls
    total_tok += u.total_tokens
    print(f"  {name:18} {u.calls:2} wyw. | {u.total_tokens:7,} tok | ${c:.4f}")
print(f"\nRAZEM: {total_calls} wywołań | {total_tok:,} tok | ${total_cost:.4f}")

Koszt pipeline'u (model: gpt-5.4-mini) — per etap:

  file_description   16 wyw. |  28,678 tok | $0.0355
  generate_tasks      1 wyw. |   5,779 tok | $0.0093
  make_task           5 wyw. |  40,691 tok | $0.0614
  generate_strategy   1 wyw. |   9,738 tok | $0.0112
  generate_document   1 wyw. |  14,849 tok | $0.0329

RAZEM: 24 wywołań | 99,735 tok | $0.1503


In [13]:
from src.output import save_appeal

appeal = document.tekst
path = save_appeal(appeal, "agent_linear")
print("Zapisano apelację do:", path)

Zapisano apelację do: /Users/ewasuknarowska/Projects/WomenInTech/data/output/apelacja_agent_linear_2026-06-06_191306.txt


## Ewaluacja

Ten notebook **tylko generuje** apelację (krok po kroku). Jak ją ocenić —
**pokrycie** i **jakość** — pokazuje osobny `eval_walkthrough.ipynb`
(na przykładzie baseline; tę samą `evaluate(...)` można wywołać na apelacji
liniowej). Pełny przebieg z oceną mocnym modelem: `python -m agent_linear.pipeline`.